# Phase 0 - free-running collapse check (no retrain; T4-friendly)

Fresh Colab, **GPU** (A100 or **free T4** both fine), then **Run All**. Reloads the already-trained **P6 two-stage adapter from HF** and drives it **free-running** (greedy vs sampling) on a real held-out row, printing decoded residual **RMS** + code **histogram**.

**T4-safe:** the base is loaded straight into 4-bit on the GPU (fits ~16GB) and the vocab is resized in-memory - no fp32 CPU load, no `base_resized` build. Compute dtype auto-falls back to fp16 on the T4 (no hardware bf16). The only real wait is the one-time ~9.85GB corpus stage (GPU-independent).

**Read the result:**
- greedy RMS ~0.001 + one dominant code  => collapse (exposure bias) confirmed
- sampling RMS ~0.05-0.2 with varied codes => signal underneath => do the incremental fixes
- sampling ALSO ~0.001 => little signal => structural pivot


In [1]:
# CELL 1 - provision the runtime (clone, install, HF token, stage the corpus)
import os, pathlib, subprocess, sys
REPO = '/content/SLM'
BRANCH = 'feat/two-stage'
if not pathlib.Path(REPO, '.git').is_dir():
    subprocess.run(['git', 'clone', 'https://github.com/ericrcwu001/SLM', REPO], check=True)
os.chdir(REPO)
if not os.environ.get('SLM_PROVISIONED'):
    subprocess.run(['git', 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', 'checkout', BRANCH], check=True)
    subprocess.run(['git', 'pull', 'origin', BRANCH], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[sft]'], check=True)
    os.environ['SLM_PROVISIONED'] = '1'
print('HEAD:', subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout.strip())

def _env_token(name):
    for _p in ('/content/SLM/.env', '/content/.env', '.env'):
        fp = pathlib.Path(_p)
        if fp.is_file():
            for _l in fp.read_text().splitlines():
                s = _l.strip()
                if s.startswith(name + '='):
                    return s.split('=', 1)[1].strip().strip('"').strip("'")
    return None
import getpass
if not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = _env_token('HF_TOKEN') or getpass.getpass('HF_TOKEN (read, for staging + adapter): ').strip()
print('HF_TOKEN set:', bool(os.environ.get('HF_TOKEN')))

os.environ['SLM_ARTIFACT_ROOT'] = '/content/slm'
if not pathlib.Path('/content/slm/tokenizer/final/model.pt').is_file():
    print('corpus missing -> staging ~9.85GB from hf://datasets/ericrcwu/LUT_SLM ...')
    subprocess.run(['slm_stage', 'stage', '--durable-root', 'hf://datasets/ericrcwu/LUT_SLM',
                    '--local-root', '/content/slm'], check=True)
else:
    print('corpus already staged at /content/slm')


HEAD: ec028d6 Add self-contained Phase 0 collapse-check notebook (T4-friendly)
HF_TOKEN set: True
corpus missing -> staging ~9.85GB from hf://datasets/ericrcwu/LUT_SLM ...


In [2]:
# CELL 2 - config + download the P6 two-stage adapter from HF (no retrain, no base_resized)
import os, json, pathlib
from huggingface_hub import login, snapshot_download

BASE_MODEL_ID = 'Qwen/Qwen2.5-VL-3B-Instruct'
REPO_ID = 'ericrcwu/LUT_SLM_sft_adapters'
ADAPTER_SUBFOLDER = 'p6_twostage_d0f9c744_smokefull'   # the P6 two-stage run (0.329 token acc)
MIN_PIXELS, MAX_PIXELS = 3136, 200704
EXPECTED_VOCAB_SIZE = 151924
EXPECTED_TOKENIZER_VERSION = 'vq_v2_srgbres_17to4_cb256_t64__w91cffdd2c82f'

login(token=os.environ['HF_TOKEN'], add_to_git_credential=False)
snapshot_download(repo_id=REPO_ID, allow_patterns=[ADAPTER_SUBFOLDER + '/*'],
                  local_dir='/content/adapters', token=os.environ['HF_TOKEN'])
adapter_dir = str(pathlib.Path('/content/adapters') / ADAPTER_SUBFOLDER)
am = json.load(open(pathlib.Path(adapter_dir) / 'adapter_manifest.json'))
assert am['base_model_id'] == BASE_MODEL_ID
assert am['vocab_size_after_resize'] == EXPECTED_VOCAB_SIZE
assert am['tokenizer_version'] == EXPECTED_TOKENIZER_VERSION
assert pathlib.Path(adapter_dir, 'adapter_model.safetensors').is_file()
print('adapter ready:', adapter_dir, '| step:', am.get('adapter_step'), '| sha:', am.get('adapter_sha256', '')[:16])


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

adapter ready: /content/adapters/p6_twostage_d0f9c744_smokefull | step: 182 | sha: 7a4f8a1a55d8dba4


In [3]:
# CELL 3 - load base in 4-bit + resize vocab IN-MEMORY + attach adapter + frozen VQ decoder (T4-safe)
import os, gc, numpy as np, torch
os.environ['SLM_ARTIFACT_ROOT'] = '/content/slm'
from transformers import AutoProcessor, AutoTokenizer, BitsAndBytesConfig, Qwen2_5_VLForConditionalGeneration
from peft import PeftModel
from tokenizer.frozen import load_frozen_vqvae

assert torch.cuda.is_available(), 'no CUDA - Runtime > Change runtime type > GPU (A100 or T4)'
gpu = torch.cuda.get_device_properties(0)
print('GPU:', gpu.name, '(%.1f GiB)' % (gpu.total_memory / 2**30))

# rebuild the exact 259-token vocab (bos/eos/unsupported + lut_000..255) as the contiguous tail
lut_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
base_vocab = len(lut_tokenizer)
canonical = ['<lut_bos>', '<lut_eos>', '<unsupported>'] + ['<lut_%03d>' % i for i in range(256)]
assert lut_tokenizer.add_special_tokens({'additional_special_tokens': canonical}) == 259
assert len(lut_tokenizer) == EXPECTED_VOCAB_SIZE, (len(lut_tokenizer), EXPECTED_VOCAB_SIZE)

processor = AutoProcessor.from_pretrained(BASE_MODEL_ID, trust_remote_code=True,
                                          min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS)
processor.tokenizer = lut_tokenizer

major, _ = torch.cuda.get_device_capability(0)
compute_dtype = torch.bfloat16 if major >= 8 else torch.float16   # T4 (7.5) -> fp16
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                         bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=compute_dtype)
base = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    BASE_MODEL_ID, quantization_config=bnb, torch_dtype=compute_dtype,
    device_map={'': 0}, low_cpu_mem_usage=True, trust_remote_code=True)
if base.get_input_embeddings().num_embeddings != len(lut_tokenizer):
    base.resize_token_embeddings(len(lut_tokenizer), mean_resizing=False)   # shape-only; PEFT restores trained rows
gen = PeftModel.from_pretrained(base, adapter_dir, is_trainable=False,
                                autocast_adapter_dtype=False, low_cpu_mem_usage=True).eval()
del base; gc.collect()
assert gen.get_input_embeddings().num_embeddings == EXPECTED_VOCAB_SIZE

vqvae, _ = load_frozen_vqvae()
proc = processor
tk = processor.tokenizer
BOS = tk.convert_tokens_to_ids('<lut_bos>')
LEOS = tk.convert_tokens_to_ids('<lut_eos>')
UNSUP = tk.convert_tokens_to_ids('<unsupported>')
MEOS = tk.eos_token_id
CODES = [tk.convert_tokens_to_ids('<lut_%03d>' % i) for i in range(256)]
ID2IDX = {c: i for i, c in enumerate(CODES)}
print('loaded P6 two-stage adapter using', compute_dtype, '| CUDA alloc %.2f GiB' % (torch.cuda.memory_allocated() / 2**30))


GPU: Tesla T4 (14.6 GiB)


config.json:   0%|          | 0.00/1.37k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/5.70k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:1348: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)


loaded P6 two-stage adapter using torch.float16 | CUDA alloc 4.64 GiB


In [4]:
# CELL 4 - Phase 0 diagnostic: greedy vs sampling on the strongest held-out row
import json, numpy as np, torch
from collections import Counter
from qwen_vl_utils import process_vision_info
from sft.example import resolve_image
from sft.holdout import is_holdout_row
from data_pipeline.attribute_spec import ground_truth_attribute_spec_text

cands = []
for line in open('data/active_sft/active_rows.jsonl'):
    r = json.loads(line)
    if (r.get('is_supported') and isinstance(r.get('target_tokens'), list)
            and len(r['target_tokens']) == 64 and r.get('measured_behavior') and is_holdout_row(r)):
        cands.append(r)
        if len(cands) >= 200:
            break
assert cands, 'no holdout supported rows found'
row = max(cands, key=lambda r: float(np.sqrt((vqvae.decode(r['target_tokens']) ** 2).mean())))
spec = ground_truth_attribute_spec_text(row)   # two-stage conditioning (matches this adapter)
img = resolve_image(row['image_path'])
print('holdout row:', row.get('id'))
print('spec:', spec)
print('image:', img)

def gen_codes(text, **samp):
    user = {'role': 'user', 'content': [{'type': 'image', 'image': img}, {'type': 'text', 'text': text}]}
    t = proc.apply_chat_template([user], tokenize=False, add_generation_prompt=True)
    imgs, vids = process_vision_info([user])
    inp = proc(text=[t], images=imgs, videos=vids, return_tensors='pt').to(gen.device)
    plen = inp['input_ids'].shape[1]
    def prefix_fn(_b, ids):
        g = ids[plen:].tolist()
        if not g:
            return [BOS, UNSUP]
        if g[0] == UNSUP:
            return [MEOS]
        n = len(g) - 1
        if n < 64:
            return CODES
        if n == 64:
            return [LEOS]
        return [MEOS]
    kw = dict(max_new_tokens=68, prefix_allowed_tokens_fn=prefix_fn)
    kw.update(dict(do_sample=True, **samp) if samp else dict(do_sample=False, num_beams=1))
    with torch.no_grad():
        out = gen.generate(**inp, **kw)
    g = out[0][plen:].tolist()
    if UNSUP in g:
        return None
    return [ID2IDX[t] for t in g if t in ID2IDX][:64]

def report(label, **samp):
    codes = gen_codes(spec, **samp)
    if codes is None:
        print('  ' + label + ': REFUSED'); return
    if len(codes) != 64:
        print('  ' + label + ': non-64 (' + str(len(codes)) + ')'); return
    rms = float(np.sqrt((vqvae.decode(codes) ** 2).mean()))
    print('  %s: RMS=%.4f  top codes=%s' % (label, rms, Counter(codes).most_common(4)))

ref = float(np.sqrt((vqvae.decode(row['target_tokens']) ** 2).mean()))
print('reference (corpus target codes): RMS=%.4f  top=%s' % (ref, Counter(row['target_tokens']).most_common(4)))
print('GREEDY:');       report('greedy')
print('SAMPLE t=0.7:'); report('sample', temperature=0.7, top_p=0.9)
print('SAMPLE t=1.0:'); report('sample', temperature=1.0, top_p=0.95)
print('')
print('Read: greedy RMS ~0.001 + one dominant code => collapse (exposure bias).')
print('      sampling RMS ~0.05-0.2 varied => signal underneath => incremental fixes.')
print('      sampling ALSO ~0.001 => little signal => structural pivot.')


holdout row: e57b1f2d0856ad55fe18e447ff8a1588ebe58fdf2ba43540e8e7f184580619ab
spec: route=grade | warmer=+7.4 darker=+14.3 less_contrast=+35.7 muted=+55.1 lifted_blacks=+3.0 lifted_shadows=+3.0 softer_highlights=+24.6 matte=+66.2 split_strength=+83.0 global_hue=88 shadow_hue=129 highlight_hue=321 sat_red=-48.2 sat_orange=-51.9 sat_yellow=-55.7 sat_green=-63.7 sat_cyan=-22.1 sat_blue=-41.3 sat_magenta=-69.2
image: /content/slm/luts/raw/ppr10k/global/40_6/before.jpg
reference (corpus target codes): RMS=0.3350  top=[(71, 8), (138, 5), (242, 4), (116, 4)]
GREEDY:


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


  greedy: RMS=0.0539  top codes=[(8, 48), (124, 14), (175, 1), (168, 1)]
SAMPLE t=0.7:
  sample: RMS=0.3228  top codes=[(76, 16), (190, 8), (86, 8), (128, 5)]
SAMPLE t=1.0:
  sample: RMS=0.0560  top codes=[(125, 6), (124, 4), (168, 3), (191, 3)]

Read: greedy RMS ~0.001 + one dominant code => collapse (exposure bias).
      sampling RMS ~0.05-0.2 varied => signal underneath => incremental fixes.
      sampling ALSO ~0.001 => little signal => structural pivot.
